# Reproducibility smoke test: 200-child sample

This notebook runs `../validation/benchmark_script.py`, **unmodified, the exact script used for the published benchmark**, against the 200-child sample in `../sample` (the `observed/` and `truth/` tables, same schema as the full dataset).

**Purpose:** show the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metrics. The full-dataset benchmark (10,000 children) reports a treated-minus-untreated restricted mean survival of 0.60 months from observed arms against 1.20 months from the counterfactual truth, a progerin AUC of 0.976 and a carotid-femoral PWV AUC of 0.560 (see the dataset README). The sample has 200 children, so some groups below hold only 70 to 115 rows and the numbers are noisy. Use them as a code check, not as a number to cite.

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, "../validation/benchmark_script.py", "../sample", "./smoke_test_results.parquet"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-2000:])
assert r.returncode == 0, r.stderr

      kind                                                                                                          label     value     n        lo        hi
0  summary       restricted mean survival in months, 60 months from the index age, treated minus untreated: observed arms -0.703297    80       NaN       NaN
1  summary                                       same quantity from the counterfactual truth (every child in both worlds)  0.890875    80       NaN       NaN
2  summary           farnesylated progerin, skin biopsy 6 months after the index age: AUC for lonafarnib at the index age  0.967199    70  0.919465  0.997917
3  summary  carotid-femoral PWV at the visit nearest to 6 months after the index age: AUC for lonafarnib at the index age  0.538614   115  0.416071  0.647889
4  summary                                                                               death within 24 months, age only  0.582254  2351  0.551526  0.615135
5  summary                                          

## Reading the output

- The treatment-effect rows rest on about 80 children, so the observed-arm estimate can even change sign. That is sampling noise at n = 200, not a property of the dataset.
- The progerin row is the stable one: a biomarker that follows the drug separates treated from untreated even in the small sample. The PWV row stays close to chance, as it does in the full dataset.
- The published, citable numbers are in `../validation/benchmark_results.parquet` and `../validation/benchmark_results_readable.csv` (full 10,000-child cohort) and in the README.

In [2]:
import pandas as pd
res = pd.read_parquet("./smoke_test_results.parquet")
res[["label", "value", "n", "lo", "hi"]].round(3)

,label,value,n,lo,hi
0,"restricted mean survival in months, 60 months ...",-0.703,80,NaN,NaN
1,same quantity from the counterfactual truth (e...,0.891,80,NaN,NaN
2,"farnesylated progerin, skin biopsy 6 months af...",0.967,70,0.919,0.998
3,carotid-femoral PWV at the visit nearest to 6 ...,0.539,115,0.416,0.648
4,"death within 24 months, age only",0.582,2351,0.552,0.615
5,"death within 24 months, age + carotid-femoral PWV",0.577,2351,0.546,0.613
6,"death within 24 months, age + PWV + weight + t...",0.582,2351,0.547,0.621
7,"death within 24 months, truth predictor: damag...",0.609,2351,0.576,0.645
